# 06 - SQL Analysis
## Credit Risk Intelligence — Canadian Banking Project

## 1. Libraries & Setup

In [1]:
# Import libraries
import pandas as pd
import sqlite3

# Load datasets
df_features = pd.read_csv('../data/cs-training-features.csv')
df_segments = pd.read_csv('../data/customer_segments.csv')
df_results = pd.read_csv('../data/model_results.csv')

# Create SQLite database
conn = sqlite3.connect('../data/credit_risk.db')

# Load dataframes into SQL tables
df_features.to_sql('customers', conn, if_exists='replace', index=False)
df_segments.to_sql('segments', conn, if_exists='replace', index=False)
df_results.to_sql('predictions', conn, if_exists='replace', index=False)

print("Database created successfully!")
print(f"Tables: customers, segments, predictions")

Database created successfully!
Tables: customers, segments, predictions


## 2. Business Queries

In [2]:
# Query 1 — Default rate by age group
query1 = """
SELECT 
    age_group,
    COUNT(*) as total_customers,
    SUM(SeriousDlqin2yrs) as total_defaults,
    ROUND(AVG(SeriousDlqin2yrs) * 100, 2) as default_rate_pct
FROM customers
GROUP BY age_group
ORDER BY default_rate_pct DESC
"""

pd.read_sql(query1, conn)

,age_group,total_customers,total_defaults,default_rate_pct
0,20-30,9980,1040,10.42
1,31-40,23465,2148,9.15
2,41-50,33821,2641,7.81
3,51-60,33279,1956,5.88
4,60+,42384,1244,2.94


In [3]:
# Query 2 — Average profile by customer segment
query2 = """
SELECT 
    cluster_name,
    COUNT(*) as total_customers,
    ROUND(AVG(MonthlyIncome), 2) as avg_income,
    ROUND(AVG(DebtRatio), 2) as avg_debt_ratio,
    ROUND(AVG(total_late_payments), 2) as avg_late_payments,
    ROUND(AVG(SeriousDlqin2yrs) * 100, 2) as default_rate_pct
FROM segments
GROUP BY cluster_name
ORDER BY avg_income DESC
"""

df_profile = pd.read_sql(query2, conn)
print(df_profile)

           cluster_name  total_customers  avg_income  avg_debt_ratio  \
0        Golden Seniors            61428     7765.37            0.30   
1              Red Zone             7772     5682.69            0.42   
2     Young & Stretched            48111     5003.51            0.41   
3  Asset Rich Cash Poor             1250     3699.36            5.35   

   avg_late_payments  default_rate_pct  
0               0.14              2.12  
1               3.64             38.57  
2               0.21              7.14  
3               0.25              4.64  


In [4]:
# --- Execução das Queries de Negócio ---

# 1. Top 10 Clientes de Maior Risco (Foco para o time de Cobrança/Prevenção)
query3 = """
SELECT actual, probability, result_type
FROM predictions
ORDER BY probability DESC
LIMIT 10
"""
top_10_risk = pd.read_sql(query3, conn)

In [5]:
# 2. Distribuição de Performance do Modelo (Métrica de saúde do algoritmo)
query4 = """
SELECT 
    result_type,
    COUNT(*) as volume,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM predictions), 2) as share_pct
FROM predictions
GROUP BY result_type
ORDER BY volume DESC
"""
model_performance = pd.read_sql(query4, conn)

In [6]:
# 3. Taxa de Default por Utilização de Crédito (Validação de Regra de Negócio)
query5 = """
SELECT 
    CASE 
        WHEN RevolvingUtilizationOfUnsecuredLines < 0.3 THEN '1. Baixa (<30%)'
        WHEN RevolvingUtilizationOfUnsecuredLines < 0.7 THEN '2. Média (30-70%)'
        WHEN RevolvingUtilizationOfUnsecuredLines <= 1.0 THEN '3. Alta (70-100%)'
        ELSE '4. Acima do Limite (>100%)'
    END as faixa_utilizacao,
    COUNT(*) as total_clientes,
    ROUND(AVG(SeriousDlqin2yrs) * 100, 2) as taxa_default_pct
FROM customers
GROUP BY faixa_utilizacao
ORDER BY faixa_utilizacao ASC
"""
utilization_analysis = pd.read_sql(query5, conn)

# --- Exibição dos Resultados ---
print("TABELA 1: Top 10 Risco Máximo")
display(top_10_risk)

print("\nTABELA 2: Performance do Modelo")
display(model_performance)

print("\nTABELA 3: Análise por Utilização de Crédito")
display(utilization_analysis)

TABELA 1: Top 10 Risco Máximo


,actual,probability,result_type
0,1,0.973469,Hit (Defaulter)
1,1,0.970936,Hit (Defaulter)
2,1,0.968922,Hit (Defaulter)
3,1,0.968790,Hit (Defaulter)
4,0,0.968739,False Alarm
5,1,0.968183,Hit (Defaulter)
6,1,0.967411,Hit (Defaulter)
7,0,0.967293,False Alarm
8,1,0.966546,Hit (Defaulter)
9,1,0.965091,Hit (Defaulter)



TABELA 2: Performance do Modelo


,result_type,volume,share_pct
0,Hit (Good Payer),22322,78.09
1,False Alarm,4458,15.60
2,Hit (Defaulter),1287,4.50
3,Miss (Hidden Default),519,1.82



TABELA 3: Análise por Utilização de Crédito


,faixa_utilizacao,total_clientes,taxa_default_pct
0,1. Baixa (<30%),89128,2.23
1,2. Média (30-70%),26522,7.41
2,3. Alta (70-100%),25490,17.43
3,4. Acima do Limite (>100%),1789,35.33
